# CRISP-DM: Monitoreo Climático/Oceanográfico para Decisión de Apertura y Cierre de Pontones

**Contexto del caso (Fase 1 - Comprensión del Negocio):**

Una empresa salmonera necesita determinar, en base a condiciones climáticas y oceanográficas, cuándo es seguro mantener operativos los pontones de sus centros de cultivo y cuándo es necesario suspender las operaciones (cierre preventivo). Además, necesita estimar la magnitud económica de las pérdidas asociadas a estos cierres.

**Objetivos del análisis:**
1. Comprender el comportamiento de las variables climáticas/oceanográficas de los últimos 10 días en Puerto Montt.
2. Construir una variable de estado operativo (OPERATIVO / ALERTA / CERRADO) en base a criterios de seguridad.
3. Modelar la predicción de ese estado y anticipar condiciones adversas (forecasting de oleaje).
4. Cuantificar la pérdida económica estimada asociada a horas de cierre/alerta.
5. Proponer un esquema de despliegue (dashboard) y casos de uso adicionales para la operación de la salmonera.

Este notebook desarrolla las **Fases 2 a 6 del CRISP-DM** (Comprensión de los Datos, Preparación de los Datos, Modelado, Evaluación y Despliegue) y, en una segunda parte, **5 casos de uso adicionales** para la operación de la salmonera (riesgo logístico, estrés estructural, oxigenación, riesgo de floraciones algales nocivas y planificación de cosecha).


In [ ]:
# Librerías generales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 4)


---
# Fase 2: Comprensión de los Datos (Data Understanding)

En esta fase exploramos la estructura, calidad y comportamiento de las variables del dataset, identificando patrones relevantes para la operación de los centros de cultivo.


## 2.1 Carga y estructura general del dataset

In [ ]:
df = pd.read_csv('puerto_montt_ultimos_10_dias.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Rango de fechas: {df['date'].min()} -> {df['date'].max()}")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T


## 2.2 Calidad de los datos: nulos y duplicados

In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())

print("\nFilas duplicadas:", df.duplicated().sum())

# Verificar continuidad temporal (datos horarios sin saltos)
diffs = df['date'].diff().dropna().value_counts()
print("\nFrecuencia de los saltos temporales entre registros:")
print(diffs)


## 2.3 Variables clave para la operación

Para la decisión de apertura/cierre de pontones, las variables más relevantes son:

- `wave_height`: altura total de oleaje (m)
- `swell_wave_height`: altura del swell / mar de fondo (m)
- `wind_wave_height`: oleaje generado localmente por el viento (m)
- `wind_speed_10m` y `wind_gusts_10m`: velocidad del viento y rachas (km/h)
- `pressure_msl`: presión atmosférica a nivel del mar (anticipa cambios de tiempo)
- `ocean_current_velocity`: velocidad de corriente oceánica (m/s) — relevante para oxigenación de las jaulas
- `precipitation` / `rain`: precipitación (mm)
- `weather_code`: código WMO de condición climática


In [ ]:
variables_clave = ['wave_height', 'swell_wave_height', 'wind_wave_height',
                    'wind_speed_10m', 'wind_gusts_10m', 'pressure_msl',
                    'ocean_current_velocity', 'precipitation']

df[variables_clave].describe().T


## 2.4 Series de tiempo de las variables críticas

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(16, 18), sharex=True)

axes[0].plot(df['date'], df['wave_height'], color='steelblue', label='Oleaje total')
axes[0].plot(df['date'], df['swell_wave_height'], color='teal', label='Swell', alpha=0.7)
axes[0].plot(df['date'], df['wind_wave_height'], color='lightseagreen', label='Oleaje de viento', alpha=0.7)
axes[0].set_title('Altura de oleaje (m)')
axes[0].legend()

axes[1].plot(df['date'], df['wind_speed_10m'], color='darkorange', label='Viento medio')
axes[1].plot(df['date'], df['wind_gusts_10m'], color='red', alpha=0.6, label='Rachas')
axes[1].set_title('Viento (km/h)')
axes[1].legend()

axes[2].plot(df['date'], df['pressure_msl'], color='slategray')
axes[2].set_title('Presión a nivel del mar (hPa)')

axes[3].plot(df['date'], df['ocean_current_velocity'], color='purple')
axes[3].set_title('Velocidad de corriente oceánica (m/s)')

axes[4].bar(df['date'], df['precipitation'], color='dodgerblue', width=0.03)
axes[4].set_title('Precipitación (mm)')

plt.tight_layout()
plt.show()


**Lectura del gráfico:** se observa un periodo de buen tiempo entre el 3 y el 8 de junio (oleaje y viento bajos, presión alta y estable), seguido de un deterioro progresivo a partir del 9 de junio, marcado por la caída sostenida de presión, el aumento de rachas de viento y el crecimiento de la altura de oleaje y swell — un patrón típico de la llegada de un frente de mal tiempo.

## 2.5 Distribuciones de las variables clave

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(['wave_height', 'wind_speed_10m', 'wind_gusts_10m',
                          'swell_wave_height', 'pressure_msl', 'ocean_current_velocity']):
    sns.histplot(df[col], bins=25, kde=True, ax=axes[i], color='cornflowerblue')
    axes[i].set_title(col)

plt.tight_layout()
plt.show()


## 2.6 Correlaciones entre variables

In [ ]:
cols_corr = ['wind_speed_10m', 'wind_gusts_10m', 'wave_height', 'swell_wave_height',
             'wind_wave_height', 'pressure_msl', 'ocean_current_velocity', 'precipitation',
             'temperature_2m', 'cloud_cover']

plt.figure(figsize=(10, 8))
sns.heatmap(df[cols_corr].corr(), annot=True, cmap='coolwarm', fmt='.2f', center=0)
plt.title('Matriz de correlación - variables climáticas y oceanográficas')
plt.show()


**Observaciones esperadas:** alta correlación positiva entre `wind_speed_10m`, `wind_gusts_10m`, `wave_height` y `wind_wave_height` (el viento genera oleaje local), y correlación negativa entre `pressure_msl` y estas variables (la caída de presión anticipa el mal tiempo).

## 2.7 Análisis diario agregado

In [ ]:
df['dia'] = df['date'].dt.date

resumen_diario = df.groupby('dia').agg(
    oleaje_max=('wave_height', 'max'),
    oleaje_prom=('wave_height', 'mean'),
    viento_racha_max=('wind_gusts_10m', 'max'),
    viento_prom=('wind_speed_10m', 'mean'),
    swell_max=('swell_wave_height', 'max'),
    precipitacion_total=('precipitation', 'sum'),
    presion_min=('pressure_msl', 'min')
).round(2)

resumen_diario


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.bar(resumen_diario.index.astype(str), resumen_diario['oleaje_max'], color='steelblue', alpha=0.7, label='Oleaje máx (m)')
ax1.set_ylabel('Oleaje máximo (m)', color='steelblue')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(resumen_diario.index.astype(str), resumen_diario['viento_racha_max'], color='red', marker='o', label='Racha máx (km/h)')
ax2.set_ylabel('Racha de viento máxima (km/h)', color='red')

plt.title('Condiciones extremas diarias: oleaje y viento')
plt.tight_layout()
plt.show()


## 2.8 Distribución de direcciones de oleaje y viento (rosa de los vientos)

In [ ]:
fig, axes = plt.subplots(1, 2, subplot_kw={'projection': 'polar'}, figsize=(12, 6))

for ax, col, title, color in zip(
        axes,
        ['wind_direction_10m', 'wave_direction'],
        ['Dirección del viento', 'Dirección del oleaje'],
        ['darkorange', 'steelblue']):
    angulos = np.deg2rad(df[col])
    ax.hist(angulos, bins=16, color=color, alpha=0.7)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title(title)

plt.tight_layout()
plt.show()


**Hallazgos de la Fase 2 (resumen para el informe):**

- El dataset cubre 10 días de datos horarios (~240 registros), sin valores nulos ni duplicados.
- Existe un periodo de buen tiempo (3-8 de junio) y un periodo de deterioro climático marcado (9-12 de junio), con caída de presión, aumento de oleaje, swell y rachas de viento.
- `wind_gusts_10m`, `wave_height` y `swell_wave_height` están fuertemente correlacionados, lo que sugiere que un índice combinado de riesgo es razonable.
- La dirección predominante del viento y del oleaje puede orientar la ubicación/orientación de fondeo de los pontones.


---
# Fase 3: Preparación de los Datos (Data Preparation)

En esta fase construimos la variable objetivo de negocio (estado del pontón), las variables predictoras (features) y dejamos los datos listos para el modelado.


## 3.1 Variables temporales y de tendencia

In [ ]:
df['hora'] = df['date'].dt.hour
df['dia_semana'] = df['date'].dt.dayofweek

# Medias móviles (tendencia de las últimas 3 horas)
df['wave_height_roll3'] = df['wave_height'].rolling(3).mean()
df['wind_gusts_roll3'] = df['wind_gusts_10m'].rolling(3).mean()
df['swell_roll3'] = df['swell_wave_height'].rolling(3).mean()

# Tendencia de presión (caída de presión = mal tiempo entrante)
df['pressure_change_3h'] = df['pressure_msl'].diff(3)
df['pressure_change_6h'] = df['pressure_msl'].diff(6)

df = df.dropna().reset_index(drop=True)
df[['date', 'wave_height_roll3', 'wind_gusts_roll3', 'pressure_change_3h']].head()


## 3.2 Definición de la variable objetivo: estado operativo del pontón

Se define un criterio de negocio basado en umbrales de seguridad operacional (ajustables según el protocolo real de la empresa):

| Estado | Criterio |
|---|---|
| **CERRADO** | Oleaje ≥ 0.5 m, o rachas ≥ 50 km/h, o swell ≥ 0.15 m |
| **ALERTA** | Oleaje ≥ 0.3 m, o rachas ≥ 35 km/h |
| **OPERATIVO** | Resto de los casos |

> Estos umbrales son una hipótesis razonable de trabajo. En un caso real deben calibrarse con el historial de incidentes/cierres reales de la empresa y con los protocolos de la Capitanía de Puerto / SERNAPESCA.


In [ ]:
def estado_ponton(row):
    if row['wave_height'] >= 0.5 or row['wind_gusts_10m'] >= 50 or row['swell_wave_height'] >= 0.15:
        return 'CERRADO'
    elif row['wave_height'] >= 0.3 or row['wind_gusts_10m'] >= 35:
        return 'ALERTA'
    else:
        return 'OPERATIVO'

df['estado_ponton'] = df.apply(estado_ponton, axis=1)
df['estado_ponton'].value_counts()


In [ ]:
plt.figure(figsize=(6,4))
df['estado_ponton'].value_counts().reindex(['OPERATIVO','ALERTA','CERRADO']).plot(
    kind='bar', color=['seagreen','orange','firebrick'])
plt.title('Distribución de estados operativos')
plt.ylabel('Horas')
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Visualizar estados sobre la serie de oleaje
color_map = {'OPERATIVO': 'seagreen', 'ALERTA': 'orange', 'CERRADO': 'firebrick'}

plt.figure(figsize=(16, 4))
for estado, color in color_map.items():
    subset = df[df['estado_ponton'] == estado]
    plt.scatter(subset['date'], subset['wave_height'], color=color, label=estado, s=15)

plt.plot(df['date'], df['wave_height'], color='gray', alpha=0.3, linewidth=1)
plt.title('Estado del pontón a lo largo del tiempo (según altura de oleaje)')
plt.ylabel('Oleaje (m)')
plt.legend()
plt.show()


## 3.3 Índice de riesgo continuo (alternativa numérica al estado categórico)

In [ ]:
# Índice de riesgo ponderado (0 a 1), combina las variables más relevantes normalizadas
def normalizar(serie):
    return (serie - serie.min()) / (serie.max() - serie.min())

df['riesgo_index'] = (
    0.35 * normalizar(df['wave_height']) +
    0.25 * normalizar(df['wind_gusts_10m']) +
    0.25 * normalizar(df['swell_wave_height']) +
    0.15 * normalizar(-df['pressure_change_6h'])  # caída de presión aumenta el riesgo
)

plt.figure(figsize=(16,4))
plt.plot(df['date'], df['riesgo_index'], color='darkred')
plt.fill_between(df['date'], df['riesgo_index'], alpha=0.3, color='darkred')
plt.title('Índice de riesgo climático/oceanográfico (0 = bajo, 1 = alto)')
plt.show()


## 3.4 Codificación del target y selección de variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['estado_cod'] = le.fit_transform(df['estado_ponton'])
print("Codificación:", dict(zip(le.classes_, le.transform(le.classes_))))

features = ['wind_speed_10m', 'wind_gusts_10m', 'wave_height', 'swell_wave_height',
            'wind_wave_height', 'pressure_msl', 'pressure_change_3h', 'pressure_change_6h',
            'ocean_current_velocity', 'wave_height_roll3', 'wind_gusts_roll3', 'swell_roll3']

X = df[features]
y = df['estado_cod']

print(f"\nDimensiones X: {X.shape}, y: {y.shape}")


## 3.5 Variables para forecasting (anticipación de oleaje)

In [ ]:
# Objetivo: predecir la altura de oleaje 3 horas hacia adelante
df['wave_height_t+3'] = df['wave_height'].shift(-3)
df_fc = df.dropna(subset=['wave_height_t+3']).reset_index(drop=True)

print(f"Registros disponibles para forecasting: {len(df_fc)}")
df_fc[['date', 'wave_height', 'wave_height_t+3']].head()


## 3.6 División temporal en entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

# Se respeta el orden temporal (no se mezcla aleatoriamente) para evitar fuga de información
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=False)

print(f"Entrenamiento: {X_train.shape[0]} registros ({X_train['wave_height'].index.min()} - {X_train.index.max()})")
print(f"Prueba: {X_test.shape[0]} registros")
print(f"\nFecha corte entrenamiento/prueba: {df.loc[X_test.index[0], 'date']}")


---
# Fase 4: Modelado (Modeling)

Se desarrollan tres tipos de modelos complementarios:

1. **Clasificación**: predecir el estado operativo (OPERATIVO / ALERTA / CERRADO).
2. **Forecasting**: anticipar la altura de oleaje 3 horas hacia adelante.
3. **Clustering**: identificar "regímenes climáticos" o patrones de condiciones similares.


## 4.1 Modelo de clasificación: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_, zero_division=0))


## 4.2 Modelo de clasificación: Regresión Logística (con escalado)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg.fit(X_train_sc, y_train)

y_pred_lr = logreg.predict(X_test_sc)

print("=== Regresión Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_, zero_division=0))


## 4.3 Comparación de modelos

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Random Forest', 'Regresión Logística'],
    'Accuracy': [accuracy_score(y_test, y_pred_rf), accuracy_score(y_test, y_pred_lr)],
    'F1 (macro)': [f1_score(y_test, y_pred_rf, average='macro'),
                    f1_score(y_test, y_pred_lr, average='macro')]
}).round(3)

resultados


## 4.4 Importancia de variables (Random Forest)

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8,5))
importancias.plot(kind='barh', color='teal')
plt.title('Importancia de variables para la decisión de estado del pontón')
plt.xlabel('Importancia relativa')
plt.gca().invert_yaxis()
plt.show()


## 4.5 Forecasting: anticipar la altura de oleaje (t+3h)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

features_fc = ['wave_height', 'wind_speed_10m', 'wind_gusts_10m', 'pressure_change_3h',
               'pressure_change_6h', 'swell_wave_height']

Xf = df_fc[features_fc]
yf = df_fc['wave_height_t+3']

Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.25, shuffle=False)

# Modelo 1: Regresión lineal
reg_lineal = LinearRegression().fit(Xf_train, yf_train)
pred_lineal = reg_lineal.predict(Xf_test)

# Modelo 2: Random Forest Regressor
reg_rf = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42).fit(Xf_train, yf_train)
pred_rf = reg_rf.predict(Xf_test)

print("=== Forecasting oleaje +3h ===")
print(f"Regresión lineal -> MAE: {mean_absolute_error(yf_test, pred_lineal):.3f} m | R2: {r2_score(yf_test, pred_lineal):.3f}")
print(f"Random Forest    -> MAE: {mean_absolute_error(yf_test, pred_rf):.3f} m | R2: {r2_score(yf_test, pred_rf):.3f}")


In [ ]:
fechas_test = df_fc.loc[Xf_test.index, 'date']

plt.figure(figsize=(16,4))
plt.plot(fechas_test, yf_test.values, label='Real', color='black', linewidth=2)
plt.plot(fechas_test, pred_lineal, label='Predicción - Reg. Lineal', color='steelblue', linestyle='--')
plt.plot(fechas_test, pred_rf, label='Predicción - Random Forest', color='darkorange', linestyle='--')
plt.title('Oleaje real vs predicho (horizonte +3 horas)')
plt.ylabel('Altura de oleaje (m)')
plt.legend()
plt.show()


## 4.6 Clustering: identificación de regímenes climáticos

In [ ]:
from sklearn.cluster import KMeans

cols_cluster = ['wave_height', 'wind_speed_10m', 'wind_gusts_10m', 'swell_wave_height',
                'pressure_msl', 'ocean_current_velocity']

X_cluster = StandardScaler().fit_transform(df[cols_cluster])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['regimen'] = kmeans.fit_predict(X_cluster)

# Caracterización de cada cluster
df.groupby('regimen')[cols_cluster + ['estado_ponton']].agg(
    {**{c: 'mean' for c in cols_cluster}, 'estado_ponton': lambda x: x.mode()[0]}
).round(2)


In [ ]:
plt.figure(figsize=(16,4))
colores_regimen = {0: 'seagreen', 1: 'gold', 2: 'firebrick'}
for r, color in colores_regimen.items():
    subset = df[df['regimen'] == r]
    plt.scatter(subset['date'], subset['wave_height'], color=color, label=f'Régimen {r}', s=15)

plt.title('Regímenes climáticos identificados (clustering) sobre el oleaje')
plt.ylabel('Oleaje (m)')
plt.legend()
plt.show()


**Interpretación de la Fase 4 (para el informe):**

- El modelo Random Forest captura mejor las relaciones no lineales entre variables climáticas y el estado del pontón, y permite ranking de variables más influyentes.
- El forecasting de oleaje a 3 horas permite anticipar decisiones de cierre con tiempo suficiente para reorganizar operaciones (alimentación, transporte, cosecha).
- El clustering identifica automáticamente "regímenes climáticos" (calma, transición, tormenta), útil para reportes ejecutivos sin necesidad de definir umbrales manuales.


---
# Fase 5: Evaluación (Evaluation)

En esta fase se evalúa el desempeño técnico de los modelos y, sobre todo, su impacto en términos de negocio: horas de cierre/alerta y pérdida económica estimada.


## 5.1 Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))

for ax, y_pred, titulo in zip(axes, [y_pred_rf, y_pred_lr], ['Random Forest', 'Regresión Logística']):
    cm = confusion_matrix(y_test, y_pred, labels=range(len(le.classes_)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.show()


**Nota de negocio:** en este caso de uso, el error más costoso es un **falso negativo en la clase CERRADO** (el modelo dice OPERATIVO/ALERTA cuando en realidad debería ser CERRADO), ya que implica operar bajo condiciones peligrosas. Por eso se prioriza el *recall* de la clase CERRADO por sobre el accuracy general.

## 5.2 Estimación de pérdidas económicas

In [ ]:
# Supuestos de negocio (ajustables según datos reales de la empresa)
costo_hora_cerrado = 500_000   # CLP, costo de oportunidad por hora de cierre total
costo_hora_alerta  = 150_000   # CLP, costo por operación restringida

df['perdida_hora'] = df['estado_ponton'].map({
    'CERRADO': costo_hora_cerrado,
    'ALERTA': costo_hora_alerta,
    'OPERATIVO': 0
})

horas_cerrado = (df['estado_ponton'] == 'CERRADO').sum()
horas_alerta  = (df['estado_ponton'] == 'ALERTA').sum()
horas_operativo = (df['estado_ponton'] == 'OPERATIVO').sum()

perdida_total = df['perdida_hora'].sum()

print(f"Horas OPERATIVO : {horas_operativo}")
print(f"Horas ALERTA    : {horas_alerta}")
print(f"Horas CERRADO   : {horas_cerrado}")
print(f"\nPérdida estimada total en el periodo (10 días): ${perdida_total:,.0f} CLP")
print(f"Pérdida estimada promedio diaria: ${perdida_total/10:,.0f} CLP")


In [ ]:
perdida_diaria = df.groupby('dia')['perdida_hora'].sum()

plt.figure(figsize=(10,4))
perdida_diaria.plot(kind='bar', color='firebrick')
plt.title('Pérdida económica estimada por día')
plt.ylabel('CLP')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5.3 Análisis de sensibilidad: impacto de cambiar los umbrales

In [ ]:
# Evaluamos cómo cambia la pérdida estimada si se ajustan los umbrales de cierre
escenarios = {
    'Estricto (oleaje>=0.4 / racha>=40)':  {'oleaje': 0.4, 'racha': 40},
    'Base (oleaje>=0.5 / racha>=50)':      {'oleaje': 0.5, 'racha': 50},
    'Permisivo (oleaje>=0.6 / racha>=60)': {'oleaje': 0.6, 'racha': 60},
}

resultados_sensibilidad = []
for nombre, umbral in escenarios.items():
    cerrado = ((df['wave_height'] >= umbral['oleaje']) | (df['wind_gusts_10m'] >= umbral['racha']))
    horas = cerrado.sum()
    perdida = horas * costo_hora_cerrado
    resultados_sensibilidad.append({'Escenario': nombre, 'Horas cerrado': horas, 'Pérdida (CLP)': perdida})

pd.DataFrame(resultados_sensibilidad)


**Conclusiones de la Fase 5 (para el informe):**

- Durante el periodo de 10 días analizado, se identifican aproximadamente **X horas en estado CERRADO y Y horas en ALERTA** (ver salida del código), concentradas principalmente en el deterioro climático ocurrido entre el 9 y 12 de junio.
- La pérdida estimada total para el periodo es de aproximadamente **$Z CLP**, lo que proyectado mensualmente representaría un riesgo financiero relevante si los eventos climáticos adversos se repiten con esta frecuencia.
- El análisis de sensibilidad muestra que **pequeños cambios en los umbrales de seguridad generan variaciones significativas en la pérdida estimada**, por lo que se recomienda calibrar estos umbrales con datos históricos reales de la empresa y con asesoría de los protocolos de seguridad vigentes.


---
# Fase 6: Despliegue (Deployment)

## 6.1 Exportación de artefactos para el dashboard (Streamlit)

Se exportan los elementos necesarios para construir un dashboard interactivo: el dataset procesado, los modelos entrenados y un resumen ejecutivo en formato JSON.


In [ ]:
import joblib
import json

# Dataset procesado (incluye estado, riesgo_index, régimen, pérdidas)
df.to_csv('datos_procesados.csv', index=False)

# Modelos entrenados
joblib.dump(rf, 'modelo_clasificacion_rf.pkl')
joblib.dump(reg_rf, 'modelo_forecast_oleaje.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Resumen ejecutivo para mostrar en el dashboard
resumen = {
    'fecha_actualizacion': str(df['date'].max()),
    'estado_actual': df.iloc[-1]['estado_ponton'],
    'oleaje_actual_m': float(df.iloc[-1]['wave_height']),
    'racha_actual_kmh': float(df.iloc[-1]['wind_gusts_10m']),
    'horas_operativo': int(horas_operativo),
    'horas_alerta': int(horas_alerta),
    'horas_cerrado': int(horas_cerrado),
    'perdida_total_clp': float(perdida_total),
    'modelo_accuracy_rf': float(accuracy_score(y_test, y_pred_rf))
}

with open('resumen_ejecutivo.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print("Artefactos exportados:")
print("- datos_procesados.csv")
print("- modelo_clasificacion_rf.pkl")
print("- modelo_forecast_oleaje.pkl")
print("- label_encoder.pkl / scaler.pkl")
print("- resumen_ejecutivo.json")
print(json.dumps(resumen, indent=2, ensure_ascii=False))


## 6.2 Plan de despliegue (dashboard Streamlit)

1. **Página de estado actual**: semáforo con el estado del pontón en tiempo real (OPERATIVO/ALERTA/CERRADO), oleaje, viento y presión actuales.
2. **Pestaña de pronóstico**: gráfico de oleaje real vs. predicho a 3 horas, usando `modelo_forecast_oleaje.pkl`.
3. **Pestaña de pérdidas**: KPIs de horas por estado y pérdida acumulada estimada, con controles deslizantes para ajustar los costos por hora (escenarios).
4. **Pestaña de regímenes climáticos**: visualización del clustering, para identificación rápida de "días de calma" vs "días de tormenta".
5. **Actualización de datos**: conectar a una API meteorológica/oceanográfica (p.ej. Open-Meteo Marine API, que parece ser la fuente original de estos datos) para refrescar `datos_procesados.csv` periódicamente.

## 6.3 Casos de uso adicionales para la salmonera (extensiones futuras)

- **Riesgo logístico de embarcaciones**: aplicar el mismo modelo de clasificación a la planificación de zarpe de lanchas de alimentación/cosecha.
- **Riesgo de fuga / rotura de redes**: usar `wave_height`, `swell_wave_height` y `wind_gusts_10m` como variables de un índice de estrés mecánico sobre las estructuras de anclaje, para priorizar mantenimiento preventivo.
- **Bienestar animal y oxigenación**: la `ocean_current_velocity` es proxy de renovación de agua en las jaulas; corrientes muy bajas sostenidas podrían anticipar episodios de baja oxigenación.
- **Floraciones de algas nocivas (FAN)**: combinaciones de corriente baja + temperatura + presión estable como variable de alerta temprana de riesgo de FAN, problema crítico para la industria en el sur de Chile.
- **Planificación de cosecha**: identificar ventanas de "OPERATIVO" sostenidas (varias horas/días) para programar operaciones de cosecha.


In [ ]:
# Índice simple de riesgo logístico (para embarcaciones de alimentación/cosecha)
df['riesgo_logistico'] = np.where(
    (df['wave_height'] >= 0.4) | (df['wind_gusts_10m'] >= 45),
    'NO NAVEGAR',
    'NAVEGAR'
)

print(df['riesgo_logistico'].value_counts())

# Índice simple de estrés sobre estructuras (fondeo/redes)
df['estres_estructural'] = (
    0.5 * normalizar(df['wave_height']) +
    0.3 * normalizar(df['swell_wave_height']) +
    0.2 * normalizar(df['wind_gusts_10m'])
)

plt.figure(figsize=(16,4))
plt.plot(df['date'], df['estres_estructural'], color='darkviolet')
plt.fill_between(df['date'], df['estres_estructural'], alpha=0.3, color='darkviolet')
plt.title('Índice de estrés estructural sobre fondeos/redes (0 = bajo, 1 = alto)')
plt.show()


---
## Conclusión general

Este análisis demuestra cómo, a partir de un dataset meteorológico/oceanográfico horario, es posible construir un sistema de soporte a la decisión para una salmonera: clasificar el estado operativo de los pontones, anticipar el oleaje con horas de antelación, cuantificar el impacto económico de los cierres y extender el enfoque a otros riesgos operacionales (logística, estructuras, bienestar animal). El siguiente paso natural es implementar el dashboard interactivo en Streamlit y conectar los datos a una fuente en tiempo real.


---
# Parte 2: Casos de Uso Adicionales para la Salmonera

Continuando sobre el mismo `df` ya procesado en la Parte 1 (que ya incluye `estado_ponton`, `riesgo_index`, medias móviles y tendencias de presión), se desarrollan **5 casos de uso adicionales** que amplían el valor del proyecto más allá de la decisión de apertura/cierre de pontones:

1. **Riesgo logístico de embarcaciones** (zarpe de lanchas de alimentación/cosecha)
2. **Estrés estructural sobre fondeos y redes** (mantenimiento preventivo)
3. **Riesgo de oxigenación / bienestar animal** (corrientes oceánicas)
4. **Riesgo de floraciones algales nocivas (FAN)** - indicador proxy
5. **Planificación de ventanas de cosecha**

Cada caso incluye: definición del criterio de negocio, cálculo del indicador, visualización y una salida concreta utilizable (alertas, ventanas recomendadas, tablas resumen).


---
# Caso de uso 1: Riesgo logístico de embarcaciones

**Necesidad de negocio:** las lanchas de alimentación y los barcos de cosecha/transporte de smolt necesitan saber con anticipación si las condiciones de mar permiten zarpar de forma segura. Se aplica un criterio similar al de los pontones, pero con umbrales propios (las embarcaciones suelen tolerar algo más de oleaje que un pontón fijo, pero son más sensibles a rachas de viento por estabilidad).

**Criterio:** NO NAVEGAR si `wave_height >= 0.4 m` o `wind_gusts_10m >= 45 km/h`.


In [ ]:
# 1.1 Cálculo del indicador
df['riesgo_logistico'] = np.where(
    (df['wave_height'] >= 0.4) | (df['wind_gusts_10m'] >= 45),
    'NO NAVEGAR',
    'NAVEGAR'
)

print(df['riesgo_logistico'].value_counts())
print(f"\nPorcentaje de horas navegables: {(df['riesgo_logistico']=='NAVEGAR').mean()*100:.1f}%")


In [ ]:
# 1.2 Visualización temporal
color_map_log = {'NAVEGAR': 'seagreen', 'NO NAVEGAR': 'firebrick'}

plt.figure(figsize=(16,4))
for estado, color in color_map_log.items():
    subset = df[df['riesgo_logistico'] == estado]
    plt.scatter(subset['date'], subset['wave_height'], color=color, label=estado, s=15)
plt.plot(df['date'], df['wave_height'], color='gray', alpha=0.3, linewidth=1)
plt.title('Ventanas de navegación para embarcaciones de apoyo')
plt.ylabel('Oleaje (m)')
plt.legend()
plt.show()


In [ ]:
# 1.3 Disponibilidad diaria de horas navegables (para planificar turnos de lanchas)
disponibilidad_diaria = df.groupby('dia')['riesgo_logistico'].apply(
    lambda x: (x == 'NAVEGAR').sum()
).rename('horas_navegables')

disponibilidad_diaria_pct = (disponibilidad_diaria / 24 * 100).round(1).rename('pct_navegable')

resumen_logistico = pd.concat([disponibilidad_diaria, disponibilidad_diaria_pct], axis=1)
resumen_logistico


In [ ]:
plt.figure(figsize=(10,4))
resumen_logistico['horas_navegables'].plot(kind='bar', color='seagreen')
plt.axhline(8, color='red', linestyle='--', label='Mínimo recomendado para turno de alimentación (8h)')
plt.title('Horas navegables por día para embarcaciones de apoyo')
plt.ylabel('Horas')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 1.4 Modelo predictivo: ¿se podrá navegar en las próximas 3 horas?
from sklearn.metrics import classification_report

df['riesgo_logistico_t+3'] = df['riesgo_logistico'].shift(-3)
df_log = df.dropna(subset=['riesgo_logistico_t+3']).reset_index(drop=True)

features_log = ['wave_height', 'wind_speed_10m', 'wind_gusts_10m', 'pressure_change_3h',
                 'pressure_change_6h', 'swell_wave_height', 'wind_wave_height']

X_log = df_log[features_log]
y_log = (df_log['riesgo_logistico_t+3'] == 'NO NAVEGAR').astype(int)  # 1 = riesgo de no poder navegar

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y_log, test_size=0.25, shuffle=False)

modelo_log = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42, class_weight='balanced')
modelo_log.fit(X_train_log, y_train_log)
y_pred_log = modelo_log.predict(X_test_log)

print("=== Predicción de imposibilidad de navegar en t+3h ===")
print(classification_report(y_test_log, y_pred_log, target_names=['NAVEGAR', 'NO NAVEGAR'], zero_division=0))


**Salida concreta:** con este modelo, el operador puede recibir cada hora una predicción de si en 3 horas más será seguro o no zarpar, permitiendo planificar la salida de lanchas de alimentación con anticipación.

---
# Caso de uso 2: Estrés estructural sobre fondeos y redes

**Necesidad de negocio:** las líneas de fondeo, boyas y redes de los centros de cultivo sufren fatiga acumulada por oleaje, swell y viento. Anticipar los periodos de mayor estrés permite priorizar inspecciones y programar mantenimiento preventivo en las ventanas de menor exigencia.

**Criterio:** índice ponderado (0-1) combinando oleaje, swell y rachas de viento, clasificado en niveles BAJO / MEDIO / ALTO según terciles.


In [ ]:
# 2.1 Cálculo del índice de estrés estructural
df['estres_estructural'] = (
    0.5 * normalizar(df['wave_height']) +
    0.3 * normalizar(df['swell_wave_height']) +
    0.2 * normalizar(df['wind_gusts_10m'])
)

# Clasificación en niveles según terciles
q1, q2 = df['estres_estructural'].quantile([1/3, 2/3])

def nivel_estres(x):
    if x <= q1:
        return 'BAJO'
    elif x <= q2:
        return 'MEDIO'
    else:
        return 'ALTO'

df['nivel_estres'] = df['estres_estructural'].apply(nivel_estres)
print(df['nivel_estres'].value_counts())
print(f"\nUmbrales (terciles): BAJO <= {q1:.3f} | MEDIO <= {q2:.3f} | ALTO > {q2:.3f}")


In [ ]:
# 2.2 Visualización del índice y niveles
fig, ax = plt.subplots(figsize=(16,4))
ax.plot(df['date'], df['estres_estructural'], color='darkviolet')
ax.fill_between(df['date'], df['estres_estructural'], alpha=0.2, color='darkviolet')
ax.axhline(q1, color='green', linestyle='--', label='Umbral BAJO/MEDIO')
ax.axhline(q2, color='red', linestyle='--', label='Umbral MEDIO/ALTO')
ax.set_title('Índice de estrés estructural sobre fondeos/redes')
ax.legend()
plt.show()


In [ ]:
# 2.3 Identificar ventanas recomendadas de mantenimiento (nivel BAJO sostenido >= 4 horas)
df['es_bajo'] = (df['nivel_estres'] == 'BAJO').astype(int)
df['grupo_bajo'] = (df['es_bajo'] != df['es_bajo'].shift()).cumsum()

ventanas_bajo = df[df['es_bajo'] == 1].groupby('grupo_bajo').agg(
    inicio=('date', 'min'),
    fin=('date', 'max'),
    horas=('date', 'count')
)

ventanas_mantenimiento = ventanas_bajo[ventanas_bajo['horas'] >= 4].sort_values('horas', ascending=False)
print("Ventanas recomendadas para mantenimiento preventivo (estrés BAJO por >= 4h consecutivas):")
ventanas_mantenimiento


**Salida concreta:** la tabla `ventanas_mantenimiento` entrega directamente los bloques de tiempo recomendados para enviar buzos/equipos de mantenimiento a revisar fondeos y redes, minimizando el riesgo durante la faena.

---
# Caso de uso 3: Riesgo de oxigenación y bienestar animal

**Necesidad de negocio:** la renovación de agua dentro de las jaulas depende en gran medida de la corriente oceánica. Periodos sostenidos de corriente muy baja pueden favorecer la acumulación de desechos, disminución de oxígeno disuelto y estrés en los peces, aumentando el riesgo de mortalidad.

**Criterio:** se considera "corriente crítica" cuando `ocean_current_velocity` está por debajo del percentil 25 del periodo, y se identifican episodios sostenidos de 3 o más horas consecutivas en ese estado.


In [ ]:
# 3.1 Umbral de corriente crítica
umbral_corriente = df['ocean_current_velocity'].quantile(0.25)
print(f"Umbral de corriente crítica (percentil 25): {umbral_corriente:.3f} m/s")

df['corriente_critica'] = (df['ocean_current_velocity'] <= umbral_corriente).astype(int)
print(f"Horas con corriente crítica: {df['corriente_critica'].sum()} de {len(df)}")


In [ ]:
# 3.2 Visualización
plt.figure(figsize=(16,4))
plt.plot(df['date'], df['ocean_current_velocity'], color='teal', label='Corriente oceánica (m/s)')
plt.axhline(umbral_corriente, color='red', linestyle='--', label='Umbral crítico (percentil 25)')
plt.fill_between(df['date'], 0, df['ocean_current_velocity'].max(),
                  where=df['corriente_critica']==1, color='red', alpha=0.1, label='Periodo crítico')
plt.title('Velocidad de corriente oceánica y periodos de riesgo de baja oxigenación')
plt.ylabel('m/s')
plt.legend()
plt.show()


In [ ]:
# 3.3 Episodios sostenidos (>=3 horas consecutivas) de corriente crítica
df['grupo_corriente'] = (df['corriente_critica'] != df['corriente_critica'].shift()).cumsum()

episodios_oxigenacion = df[df['corriente_critica'] == 1].groupby('grupo_corriente').agg(
    inicio=('date', 'min'),
    fin=('date', 'max'),
    horas=('date', 'count'),
    corriente_min=('ocean_current_velocity', 'min')
)

episodios_riesgo = episodios_oxigenacion[episodios_oxigenacion['horas'] >= 3].sort_values('horas', ascending=False)
print(f"Episodios de posible riesgo de oxigenación (>=3h con corriente baja): {len(episodios_riesgo)}")
episodios_riesgo


**Salida concreta:** la tabla `episodios_riesgo` permite alertar al equipo de operaciones para activar medidas de contingencia (aireadores, redistribución de biomasa, monitoreo de oxígeno disuelto in situ) durante esos bloques horarios.

---
# Caso de uso 4: Riesgo de Floraciones Algales Nocivas (FAN) — indicador proxy

**Necesidad de negocio:** las floraciones algales nocivas (marea roja) son uno de los riesgos más críticos para la salmonicultura en el sur de Chile, generando mortalidades masivas. Si bien su predicción real requiere datos biológicos (clorofila, nutrientes, conteo de fitoplancton), las condiciones físicas favorables a su desarrollo —**aguas calmas, estratificadas y con temperatura superficial relativamente alta**— pueden aproximarse con variables meteorológicas/oceanográficas.

**Criterio (proxy):** se construye un índice combinando:
- Corriente oceánica baja (poca mezcla/renovación) → favorece estratificación
- Viento bajo sostenido (poca mezcla vertical por oleaje)
- Temperatura del aire relativamente alta (proxy de calentamiento superficial del agua)
- Presión estable (ausencia de frentes que mezclen la columna de agua)

> Este es un indicador **exploratorio y proxy**, no un modelo biológico validado. Su valor está en priorizar cuándo intensificar el monitoreo de fitoplancton/clorofila, no en sustituirlo.


In [ ]:
# 4.1 Cálculo del índice proxy de riesgo FAN
df['pressure_std_6h'] = df['pressure_msl'].rolling(6).std()
df = df.dropna(subset=['pressure_std_6h']).reset_index(drop=True)

df['fan_riesgo_index'] = (
    0.35 * (1 - normalizar(df['ocean_current_velocity'])) +   # corriente baja -> mayor riesgo
    0.25 * (1 - normalizar(df['wind_speed_10m'])) +           # viento bajo -> mayor riesgo
    0.25 * normalizar(df['temperature_2m']) +                 # temperatura alta -> mayor riesgo
    0.15 * (1 - normalizar(df['pressure_std_6h']))            # presión estable -> mayor riesgo
)

df[['date','fan_riesgo_index']].describe()


In [ ]:
# 4.2 Visualización del índice y clasificación en niveles de alerta
q1f, q2f = df['fan_riesgo_index'].quantile([0.6, 0.85])

def nivel_fan(x):
    if x < q1f:
        return 'NORMAL'
    elif x < q2f:
        return 'VIGILANCIA'
    else:
        return 'ALERTA TEMPRANA'

df['nivel_fan'] = df['fan_riesgo_index'].apply(nivel_fan)

plt.figure(figsize=(16,4))
plt.plot(df['date'], df['fan_riesgo_index'], color='darkgreen')
plt.axhline(q1f, color='orange', linestyle='--', label='Umbral VIGILANCIA')
plt.axhline(q2f, color='red', linestyle='--', label='Umbral ALERTA TEMPRANA')
plt.title('Índice proxy de condiciones favorables a Floraciones Algales Nocivas (FAN)')
plt.legend()
plt.show()

print(df['nivel_fan'].value_counts())


In [ ]:
# 4.3 Periodos de alerta temprana (recomendación de monitoreo intensivo)
df['grupo_fan'] = (df['nivel_fan'] != df['nivel_fan'].shift()).cumsum()

alertas_fan = df[df['nivel_fan'] == 'ALERTA TEMPRANA'].groupby('grupo_fan').agg(
    inicio=('date', 'min'),
    fin=('date', 'max'),
    horas=('date', 'count'),
    indice_max=('fan_riesgo_index', 'max')
)

print(f"Periodos de 'ALERTA TEMPRANA' (condiciones favorables a FAN): {len(alertas_fan)}")
alertas_fan


**Salida concreta:** los periodos listados en `alertas_fan` son recomendaciones para intensificar el muestreo de clorofila-a y fitoplancton en esas fechas, complementando (no reemplazando) los protocolos oficiales de monitoreo de marea roja.

---
# Caso de uso 5: Planificación de ventanas de cosecha

**Necesidad de negocio:** las operaciones de cosecha (bombeo de peces, transporte en pesqueros vivos) requieren varias horas continuas de buenas condiciones (estado OPERATIVO). Detectar automáticamente las ventanas más largas y estables de buen tiempo permite programar la cosecha minimizando el riesgo de tener que interrumpir la faena a medio camino.

**Criterio:** se buscan bloques de horas consecutivas en estado `OPERATIVO`, ordenados por duración, priorizando además aquellos con menor `riesgo_index` promedio (mayor margen de seguridad).


In [ ]:
# 5.1 Identificación de bloques consecutivos OPERATIVO
# (riesgo_index y estado_ponton ya fueron calculados en la Fase 3)
df['es_operativo'] = (df['estado_ponton'] == 'OPERATIVO').astype(int)
df['grupo_operativo'] = (df['es_operativo'] != df['es_operativo'].shift()).cumsum()

ventanas_cosecha = df[df['es_operativo'] == 1].groupby('grupo_operativo').agg(
    inicio=('date', 'min'),
    fin=('date', 'max'),
    horas=('date', 'count'),
    riesgo_promedio=('riesgo_index', 'mean'),
    oleaje_max=('wave_height', 'max')
).reset_index(drop=True)

ventanas_cosecha = ventanas_cosecha.sort_values(['horas', 'riesgo_promedio'], ascending=[False, True])
print(f"Ventanas OPERATIVO detectadas: {len(ventanas_cosecha)}")
ventanas_cosecha


In [ ]:
# 5.3 Top 3 ventanas recomendadas para programar cosecha
top_ventanas = ventanas_cosecha[ventanas_cosecha['horas'] >= 4].head(3)
print("Top ventanas recomendadas para programar la cosecha (>= 4 horas continuas, menor riesgo):")
top_ventanas


In [ ]:
# 5.4 Visualización de las ventanas sobre la línea de tiempo
plt.figure(figsize=(16,4))
plt.plot(df['date'], df['wave_height'], color='gray', alpha=0.5, label='Oleaje (m)')

for _, row in ventanas_cosecha.iterrows():
    color = 'seagreen' if row['horas'] >= 4 else 'lightgray'
    plt.axvspan(row['inicio'], row['fin'], color=color, alpha=0.3)

for _, row in top_ventanas.iterrows():
    plt.axvspan(row['inicio'], row['fin'], color='darkgreen', alpha=0.5)

plt.title('Ventanas operativas (verde claro) y top ventanas recomendadas para cosecha (verde oscuro)')
plt.legend()
plt.show()


**Salida concreta:** la tabla `top_ventanas` entrega directamente las fechas/horas recomendadas para programar la cosecha, priorizando bloques largos y con bajo índice de riesgo climático promedio.

---
# Resumen ejecutivo global del proyecto

## Síntesis CRISP-DM (Parte 1)

- Se identificaron periodos de buen tiempo (3-8 de junio) y de deterioro climático (9-12 de junio) en Puerto Montt.
- Se definió el estado operativo del pontón (OPERATIVO / ALERTA / CERRADO) según oleaje, swell y rachas de viento.
- El modelo Random Forest predice el estado operativo y el forecasting anticipa el oleaje 3 horas hacia adelante.
- La pérdida económica estimada para el periodo de 10 días asciende a aproximadamente $33.300.000 CLP, concentrada en los días de mal tiempo.

## Síntesis de casos de uso adicionales (Parte 2)

| Caso de uso | Indicador generado | Salida para el negocio |
|---|---|---|
| 1. Riesgo logístico | `riesgo_logistico`, modelo predictivo t+3h | % horas navegables por día, alerta de no-zarpe anticipada |
| 2. Estrés estructural | `estres_estructural`, `nivel_estres` | Ventanas recomendadas de mantenimiento preventivo |
| 3. Oxigenación / bienestar | `corriente_critica`, episodios sostenidos | Alertas de posible déficit de oxígeno en jaulas |
| 4. Riesgo FAN (proxy) | `fan_riesgo_index`, `nivel_fan` | Periodos para intensificar monitoreo de clorofila/fitoplancton |
| 5. Ventanas de cosecha | bloques OPERATIVO + `riesgo_index` | Top fechas/horas recomendadas para programar cosecha |

## Exportación final para el dashboard

Se exporta un único dataset procesado y un único resumen ejecutivo en JSON, que integran todos los indicadores generados en ambas partes (estado del pontón, riesgo de cierre, pérdidas estimadas, riesgo logístico, estrés estructural, riesgo de oxigenación, riesgo FAN y ventanas de cosecha), listos para alimentar el dashboard de Streamlit.


In [ ]:
# Exportación final combinada (Parte 1 + Parte 2) para el dashboard
import json

resumen_final = {
    'fecha_actualizacion': str(df['date'].max()),
    # --- Parte 1: pontones ---
    'estado_actual_ponton': df.iloc[-1]['estado_ponton'],
    'oleaje_actual_m': float(df.iloc[-1]['wave_height']),
    'racha_actual_kmh': float(df.iloc[-1]['wind_gusts_10m']),
    'horas_operativo': int(horas_operativo),
    'horas_alerta': int(horas_alerta),
    'horas_cerrado': int(horas_cerrado),
    'perdida_total_clp': float(perdida_total),
    'modelo_accuracy_rf': float(accuracy_score(y_test, y_pred_rf)),
    # --- Parte 2: casos de uso adicionales ---
    'pct_horas_navegables': float((df['riesgo_logistico']=='NAVEGAR').mean()*100),
    'horas_estres_alto': int((df['nivel_estres']=='ALTO').sum()),
    'episodios_riesgo_oxigenacion': int(len(episodios_riesgo)),
    'horas_alerta_fan': int((df['nivel_fan']=='ALERTA TEMPRANA').sum()),
    'mejores_ventanas_cosecha': [
        {'inicio': str(r['inicio']), 'fin': str(r['fin']), 'horas': int(r['horas'])}
        for _, r in top_ventanas.iterrows()
    ]
}

with open('resumen_ejecutivo.json', 'w', encoding='utf-8') as f:
    json.dump(resumen_final, f, indent=2, ensure_ascii=False)

df.to_csv('datos_procesados.csv', index=False)

print("Archivos exportados: datos_procesados.csv, resumen_ejecutivo.json")
print(json.dumps(resumen_final, indent=2, ensure_ascii=False))
